# lipshift, Step 0b: Study A go / no-go (English vs Spanish lip-reading encoders)

Study A compares two encoders from the same codebase and architecture family
(Ma et al., 2022, "Visual Speech Recognition for Multiple Languages in the Wild"):

| Model | Upstream WER | Zoo size | Where in the README |
|---|---|---|---|
| English visual-only, LRS3 | 32.3 | 186 MB | LRS3 table (second one), Visual-only |
| Spanish visual-only, CMU-MOSEAS | 44.5 | 186 MB | CMU-MOSEAS table, Visual-only, Spanish |

Before running:
1. Open https://github.com/mpc001/Visual_Speech_Recognition_for_Multiple_Languages, section Model Zoo.
2. Download those two "Visual-only" models (GoogleDrive links). Do not download the language models or landmarks.
3. Upload both files, unchanged, to `MyDrive/lipshift/weights/multilingual/`.
4. Runtime > T4 GPU. Run Cell 1, then Runtime > Restart session, then the rest.

License: the multilingual code and models are for non-commercial research and benchmarking only.

In [ ]:
# Cell 1: installs. Then Runtime > Restart session.
!git clone -q https://github.com/mpc001/auto_avsr /content/auto_avsr || true
!git -C /content/auto_avsr checkout -q 182b62837773ab01052d4ac21ef1d2203ea7d267
!git clone -q https://github.com/mpc001/Visual_Speech_Recognition_for_Multiple_Languages /content/vsrml || true
!git -C /content/vsrml checkout -q 5e1405db0ae816509fb312f9a578724c2e0de0c7
!pip install -q mediapipe==1.2.2 sentencepiece scikit-image six
!wget -q -nc https://storage.googleapis.com/mediapipe-models/face_detector/blaze_face_short_range/float16/1/blaze_face_short_range.tflite -O /content/blaze_face_short_range.tflite
!ls -l /content/blaze_face_short_range.tflite
print("Done. Now: Runtime > Restart session, then run Cell 2.")

In [ ]:
# Cell 2: Drive, copy the model archives to local disk, unpack and verify every file
import json, sys, shutil, zipfile, tarfile
from pathlib import Path
import numpy as np, torch
from google.colab import drive
drive.mount("/content/drive", force_remount=True)
WORK = Path("/content/drive/MyDrive/lipshift")
OUT = WORK / "step0"; OUT.mkdir(parents=True, exist_ok=True)
ML_DIR = WORK / "weights" / "multilingual"
LOCAL = Path("/content/ml_zips"); LOCAL.mkdir(exist_ok=True)
UNPACK = Path("/content/ml_models"); UNPACK.mkdir(exist_ok=True)
AUTO_AVSR, VSRML = "/content/auto_avsr", "/content/vsrml"
FACE_MODEL = "/content/blaze_face_short_range.tflite"
assert torch.cuda.is_available(), "Runtime > Change runtime type > T4 GPU"
# Reading a zip directly from the Drive mount can drop the connection (Errno 107):
# copy each archive to local disk in one sequential read, then work locally.
for p in sorted(x for x in ML_DIR.glob("*.zip") if x.is_file()):
    local = LOCAL / p.name
    if not local.exists() or local.stat().st_size != p.stat().st_size:
        shutil.copyfile(p, local)
report = {}
for z in sorted(LOCAL.glob("*.zip")):
    zf = zipfile.ZipFile(z)
    bad = zf.testzip()
    if bad is not None:
        raise RuntimeError(f"{z.name} corrupted at {bad}: !rm -f /content/ml_zips/*.zip and rerun")
    dst = UNPACK / z.stem
    incomplete = [m.filename for m in zf.infolist() if not m.is_dir()
                  and (not (dst / m.filename).exists() or (dst / m.filename).stat().st_size != m.file_size)]
    if incomplete:
        shutil.rmtree(dst, ignore_errors=True)
        zf.extractall(dst)
    report[z.name] = [(str(q.relative_to(UNPACK)), round(q.stat().st_size / 1e6, 1)) for q in dst.rglob("model.pth")]
print(json.dumps(report, indent=2))
confs = sorted(UNPACK.rglob("model.json"))
def find(tag):
    hits = [c for c in confs if tag in str(c)]
    return hits[0] if hits else None
CONF = {"en": find("LRS3"), "es": find("_ES_") or find("/es/")}
print(CONF)
assert all(CONF.values()), "Could not find both model.json files: send me the printed report"

In [ ]:
%%writefile /content/step0_helpers.py
"""Step 0 helpers: video loading, mouth cropping and VSR feature extraction.

Wraps the official Auto-AVSR code (https://github.com/mpc001/auto_avsr,
Apache-2.0) with two fixes for current library versions:

1. Video is read with OpenCV, because ``torchvision.io.read_video`` is not
   present in torchvision 0.29.
2. Face keypoints come from the MediaPipe Tasks ``FaceDetector``, because the
   legacy ``mediapipe.solutions`` API used by Auto-AVSR is absent from mediapipe
   0.10.30+ and 1.x. Requires a BlazeFace short-range ``.tflite`` model file.

Pretrained weights are not covered by the Apache-2.0 code license: see the
model zoo terms (non-commercial, benchmarking use).
"""

from __future__ import annotations

import logging
import sys
import time
from pathlib import Path

import cv2
import numpy as np
import torch

logger = logging.getLogger(__name__)


def load_video_rgb(
    path: str | Path, max_frames: int | None = None
) -> tuple[np.ndarray, float]:
    """Read a video file into an RGB uint8 array of shape (T, H, W, 3).

    Args:
        path: Video file path.
        max_frames: Stop after this many frames (None reads all).

    Returns:
        Frames and the frame rate reported by the container.
    """
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        raise OSError(f"Cannot open video: {path}")
    fps = float(cap.get(cv2.CAP_PROP_FPS)) or 25.0
    frames: list[np.ndarray] = []
    while True:
        ok, frame_bgr = cap.read()
        if not ok:
            break
        frames.append(cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB))
        if max_frames is not None and len(frames) >= max_frames:
            break
    cap.release()
    if not frames:
        raise ValueError(f"No frames decoded from: {path}")
    return np.stack(frames), fps


def add_auto_avsr_to_path(auto_avsr_dir: str | Path) -> None:
    """Make the cloned Auto-AVSR repository importable."""
    p = str(Path(auto_avsr_dir).resolve())
    if p not in sys.path:
        sys.path.insert(0, p)


class TasksLandmarksDetector:
    """Drop-in replacement for Auto-AVSR's mediapipe ``LandmarksDetector``.

    Uses the MediaPipe Tasks ``FaceDetector`` (mediapipe 0.10.30+ and 1.x), which
    replaced the removed ``mediapipe.solutions`` API. BlazeFace returns six
    keypoints in the same order as the legacy API (right eye, left eye, nose tip,
    mouth center, right ear, left ear); Auto-AVSR uses the first four.

    Differences from the original: only the short-range BlazeFace model is used
    (the legacy code tried full range first), and the largest face is selected
    by bounding-box area.
    """

    def __init__(self, model_path: str | Path, min_confidence: float = 0.5) -> None:
        from mediapipe.tasks import python as mp_tasks
        from mediapipe.tasks.python import vision

        options = vision.FaceDetectorOptions(
            base_options=mp_tasks.BaseOptions(model_asset_path=str(model_path)),
            min_detection_confidence=min_confidence,
        )
        self._detector = vision.FaceDetector.create_from_options(options)

    def close(self) -> None:
        """Release the detector explicitly (avoids noisy errors at interpreter exit)."""
        if self._detector is not None:
            self._detector.close()
            self._detector = None

    def __call__(self, video_frames: np.ndarray) -> list[np.ndarray | None]:
        import mediapipe as mp

        landmarks: list[np.ndarray | None] = []
        for frame in video_frames:
            h, w = frame.shape[:2]
            image = mp.Image(
                image_format=mp.ImageFormat.SRGB, data=np.ascontiguousarray(frame)
            )
            detections = self._detector.detect(image).detections
            if not detections:
                landmarks.append(None)
                continue
            best = max(
                detections, key=lambda d: d.bounding_box.width * d.bounding_box.height
            )
            pts = [[int(k.x * w), int(k.y * h)] for k in best.keypoints[:4]]
            landmarks.append(np.array(pts))
        if all(lm is None for lm in landmarks):
            raise AssertionError("Cannot detect any frames in the video")
        return landmarks


def crop_mouths(
    frames_rgb: np.ndarray, auto_avsr_dir: str | Path, face_model_path: str | Path
) -> np.ndarray:
    """Detect the face per frame and return aligned mouth crops (T, 96, 96, 3).

    Raises:
        AssertionError: if no face is found in any frame.
        RuntimeError: if cropping returns nothing.
    """
    add_auto_avsr_to_path(auto_avsr_dir)
    from preparation.detectors.mediapipe.video_process import VideoProcess

    detector = TasksLandmarksDetector(face_model_path)
    try:
        landmarks = detector(frames_rgb)
    finally:
        detector.close()
    mouths = VideoProcess(convert_gray=False)(frames_rgb, landmarks)
    if mouths is None:
        raise RuntimeError("Mouth cropping returned None (no usable landmarks).")
    return np.asarray(mouths)


class VSREncoder(torch.nn.Module):
    """Auto-AVSR visual front-end + conformer encoder, without the decoder."""

    def __init__(self, auto_avsr_dir: str | Path) -> None:
        super().__init__()
        add_auto_avsr_to_path(auto_avsr_dir)
        from datamodule.transforms import TextTransform
        from espnet.nets.pytorch_backend.e2e_asr_conformer import E2E

        n_tokens = len(TextTransform().token_list)
        self.model = E2E(n_tokens, "video", ctc_weight=0.1)

    def load_checkpoint(self, ckpt_path: str | Path) -> dict[str, list[str]]:
        """Load weights non-strictly and report key mismatches."""
        state = torch.load(str(ckpt_path), map_location="cpu")
        if isinstance(state, dict) and "state_dict" in state:
            state = state["state_dict"]
        result = self.model.load_state_dict(state, strict=False)
        return {
            "missing": list(result.missing_keys),
            "unexpected": list(result.unexpected_keys),
        }

    @torch.inference_mode()
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Map (T, 1, 88, 88) normalized mouth frames to (T, D) features."""
        y = self.model.frontend(x.unsqueeze(0))
        y = self.model.proj_encoder(y)
        y, _ = self.model.encoder(y, None)
        return y.squeeze(0)


def preprocess_mouths(
    mouths_rgb: np.ndarray, auto_avsr_dir: str | Path
) -> torch.Tensor:
    """Apply the official test-time transform: /255, center crop 88, gray, normalize."""
    add_auto_avsr_to_path(auto_avsr_dir)
    from datamodule.transforms import VideoTransform

    video = torch.from_numpy(mouths_rgb).permute(0, 3, 1, 2)  # (T, C, H, W)
    return VideoTransform("test")(video)


def extract_features(
    video_path: str | Path,
    encoder: VSREncoder,
    auto_avsr_dir: str | Path,
    face_model_path: str | Path,
    device: str = "cpu",
    max_frames: int | None = 250,
) -> tuple[np.ndarray, dict[str, float]]:
    """Full path: video file -> mouth crops -> (T, D) float16 features and timings."""
    t0 = time.perf_counter()
    frames, fps = load_video_rgb(video_path, max_frames=max_frames)
    t1 = time.perf_counter()
    mouths = crop_mouths(frames, auto_avsr_dir, face_model_path)
    t2 = time.perf_counter()
    x = preprocess_mouths(mouths, auto_avsr_dir).to(device)
    feats = encoder.to(device).eval()(x).float().cpu().numpy().astype(np.float16)
    t3 = time.perf_counter()
    info = {
        "n_frames": float(frames.shape[0]),
        "fps": fps,
        "sec_decode": t1 - t0,
        "sec_crop": t2 - t1,
        "sec_encode": t3 - t2,
    }
    return feats, info


# --------------------------------------------------------------------------------------
# Frame-rate normalization (both model families are configured for 25 fps input)
# --------------------------------------------------------------------------------------


def resample_to_fps(
    frames: np.ndarray, fps: float, target_fps: float = 25.0
) -> np.ndarray:
    """Pick frames on an even grid so the clip plays at ``target_fps``.

    Same rule as the multilingual VSR repo (``torch.linspace`` index selection with
    ``speed_rate = fps / target_fps``), applied to raw frames before cropping.
    """
    speed_rate = fps / target_fps
    if abs(speed_rate - 1.0) < 1e-3:
        return frames
    n_out = int(frames.shape[0] / speed_rate)
    idx = torch.linspace(0, frames.shape[0] - 1, n_out, dtype=torch.int64).numpy()
    return frames[idx]


# --------------------------------------------------------------------------------------
# Multilingual VSR models (Ma et al., 2022; repo mpc001/Visual_Speech_Recognition_for_
# Multiple_Languages). This repo also ships a top-level package named ``espnet`` that
# differs from Auto-AVSR's, so imports are done in an isolated module namespace.
# --------------------------------------------------------------------------------------

_REPO_PACKAGES = ("espnet", "datamodule", "preparation", "pipelines")


def switch_repo(repo_dir: str | Path) -> None:
    """Purge cached repo packages from ``sys.modules`` and put ``repo_dir`` first on the path.

    Encoders that were already built keep working: their classes stay referenced.
    """
    for name in list(sys.modules):
        if name.split(".")[0] in _REPO_PACKAGES:
            del sys.modules[name]
    p = str(Path(repo_dir).resolve())
    sys.path[:] = [x for x in sys.path if x != p]
    sys.path.insert(0, p)


class MultilingualVSREncoder(torch.nn.Module):
    """Encoder of a model from the multilingual VSR model zoo, built from its model.json."""

    def __init__(
        self, repo_dir: str | Path, model_conf: str | Path | None = None, args=None
    ) -> None:
        super().__init__()
        import argparse
        import json

        switch_repo(repo_dir)
        from espnet.nets.pytorch_backend.e2e_asr_transformer import E2E

        if args is None:
            with open(model_conf, "rb") as f:
                confs = json.load(f)
            args = argparse.Namespace(
                **(confs if isinstance(confs, dict) else confs[2])
            )
            labels_type = getattr(args, "labels_type", "char")
            if labels_type == "char":
                odim = len(args.char_list)
            else:  # unigram5000: <blank> + 5000 units + <eos>, as in pipelines/model.py
                tok = Path(repo_dir) / "pipelines" / "tokens" / "unigram5000_units.txt"
                odim = len(tok.read_text().splitlines()) + 2
        else:
            odim = 10  # test path only
        self.args = args
        self.model = E2E(odim, args)

    def load_checkpoint(self, ckpt_path: str | Path) -> dict[str, list[str]]:
        """Load weights non-strictly and report key mismatches."""
        state = torch.load(str(ckpt_path), map_location="cpu")
        result = self.model.load_state_dict(state, strict=False)
        return {
            "missing": list(result.missing_keys),
            "unexpected": list(result.unexpected_keys),
        }

    @torch.inference_mode()
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Map (1, T, 88, 88) normalized gray mouth frames to (T, D) features."""
        self.model.eval()
        return self.model.encode(x)


def preprocess_mouths_gray(mouths_rgb: np.ndarray) -> torch.Tensor:
    """Multilingual-repo transform: gray, /255, center crop 88, normalize -> (1, T, 88, 88).

    The original converts to gray before the affine warp; we convert the warped RGB
    crop. Both are linear operations, so the difference is limited to interpolation.
    """
    import torchvision

    gray = np.stack([cv2.cvtColor(f, cv2.COLOR_RGB2GRAY) for f in mouths_rgb])
    x = torch.from_numpy(gray).float().unsqueeze(0) / 255.0  # (1, T, H, W)
    x = torchvision.transforms.CenterCrop(88)(x)
    return torchvision.transforms.Normalize(0.421, 0.165)(x)


def extract_features_v2(
    video_path: str | Path,
    encoder: torch.nn.Module,
    auto_avsr_dir: str | Path,
    face_model_path: str | Path,
    device: str = "cpu",
    max_frames: int | None = 300,
    target_fps: float = 25.0,
) -> tuple[np.ndarray, dict[str, float]]:
    """Video -> 25 fps -> mouth crops -> encoder features (T, D) float16, for either family."""
    t0 = time.perf_counter()
    frames, fps = load_video_rgb(video_path, max_frames=max_frames)
    frames = resample_to_fps(frames, fps, target_fps)
    t1 = time.perf_counter()
    switch_repo(auto_avsr_dir)  # cropper lives in Auto-AVSR
    mouths = crop_mouths(frames, auto_avsr_dir, face_model_path)
    t2 = time.perf_counter()
    if isinstance(encoder, MultilingualVSREncoder):
        x = preprocess_mouths_gray(mouths).to(device)
    else:
        x = preprocess_mouths(mouths, auto_avsr_dir).to(device)
    feats = encoder.to(device).eval()(x).float().cpu().numpy().astype(np.float16)
    t3 = time.perf_counter()
    info = {
        "fps_in": fps,
        "n_frames_25fps": float(frames.shape[0]),
        "sec_decode": t1 - t0,
        "sec_crop": t2 - t1,
        "sec_encode": t3 - t2,
    }
    return feats, info

In [ ]:
# Cell 4: build both encoders from their model.json and load weights
sys.path.insert(0, "/content")
import step0_helpers as h
enc, keys, cfg = {}, {}, {}
for lang, conf in CONF.items():
    enc[lang] = h.MultilingualVSREncoder(VSRML, model_conf=conf)
    keys[lang] = enc[lang].load_checkpoint(conf.parent / "model.pth")
    a = enc[lang].args
    cfg[lang] = {k: getattr(a, k, None) for k in ["adim", "aheads", "elayers", "eunits",
                                                  "transformer_input_layer", "labels_type"]}
    cfg[lang]["params_M"] = round(sum(p.numel() for p in enc[lang].parameters()) / 1e6, 1)
    print(lang, cfg[lang], "| missing:", len(keys[lang]["missing"]), "unexpected:", len(keys[lang]["unexpected"]))
    print("   first missing:", keys[lang]["missing"][:3], "first unexpected:", keys[lang]["unexpected"][:3])

In [ ]:
# Cell 5: pick 2 English + 2 Spanish real clips from MAVOS-DD.
# Needs a Colab secret NAMED "HF_TOKEN" (key icon). Pass the name below, never the token itself.
from google.colab import userdata
from huggingface_hub import snapshot_download, hf_hub_download
from datasets import load_from_disk
REPO, HF_TOKEN = "unibuc-cs/MAVOS-DD", userdata.get("HF_TOKEN")
META_DIR = Path("/content/mavos_meta")
snapshot_download(REPO, repo_type="dataset", local_dir=META_DIR, token=HF_TOKEN,
                  ignore_patterns=["*.mp4", "*.MP4", "*.wav"])
cand = sorted({p.parent for p in META_DIR.rglob("state.json")}) or [META_DIR]
df = load_from_disk(str(cand[0])).to_pandas()
rng = np.random.default_rng(1)
clips = {}
for lang in ["english", "spanish"]:
    sub = df[(df["language"] == lang) & (df["label"] == "real")]
    for i, j in enumerate(rng.permutation(len(sub))[:2]):
        vp = sub.iloc[j]["video_path"]
        clips[f"{lang}_{i}"] = hf_hub_download(REPO, filename=vp, repo_type="dataset",
                                               local_dir="/content/clips", token=HF_TOKEN)
print(clips)

In [ ]:
# Cell 6: extract with both encoders on all clips (25 fps), then summarize
torch.cuda.reset_peak_memory_stats()
res = {}
for clip, path in clips.items():
    for lang, e in enc.items():
        try:
            f, info = h.extract_features_v2(path, e, AUTO_AVSR, FACE_MODEL, device="cuda")
            res[f"{clip}|enc_{lang}"] = {"shape": list(f.shape), "finite": bool(np.isfinite(f).all()),
                                         **{k: round(v, 3) for k, v in info.items()}}
        except Exception as ex:  # noqa: BLE001
            res[f"{clip}|enc_{lang}"] = {"error": f"{type(ex).__name__}: {ex}"}
        print(clip, lang, res[f"{clip}|enc_{lang}"])
ok = [v for v in res.values() if "error" not in v and v["finite"]]
summary = {"configs": cfg,
           "missing_keys": {k: len(v["missing"]) for k, v in keys.items()},
           "unexpected_keys": {k: len(v["unexpected"]) for k, v in keys.items()},
           "extractions": res, "peak_gpu_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2)}
summary["GO_study_A"] = bool(all(n == 0 for n in summary["missing_keys"].values()) and len(ok) >= 6)
(OUT / "study_a_report.json").write_text(json.dumps(summary, indent=2, default=str))
print(json.dumps(summary, indent=2, default=str))